In [30]:
%%capture
%pip install -r requirements-common.txt

In [1]:
import json
import os

# Generic utils
from examples.utils import setup_notebook

# Check environment setup
if not setup_notebook(required_keys=["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]):
    raise OSError("Please set up required environment variables")

In [2]:
from urllib.parse import quote as url_quote

from requests import delete, get, post
from websockets import ConnectionClosed, connect

In [3]:
response = post("http://localhost:8000/api/v2/agents", json={
    "mode": "conversational",
    "name": "Test Agent 2: examples/create-agent.ipynb",
    "version": "1.0.0",
    "description": (
        "This is a test agent created by the examples/create-agent.ipynb script."
    ),
    "runbook_raw_text": "# Objective\nYou are a helpful assistant.",
    "platform_configs": [{
        "kind": "bedrock",
        "region_name": os.environ["AWS_DEFAULT_REGION"],
        "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"],
        "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    }],
    "action_packages": [{
        "name": "browsing",
        "version": "1.0.0",
        "organization": "testorg",
        "url": "localhost:31415",
        "api_key": { "value": "some-api-key" },
        "allowed_actions": ["search", "browse"],
    }],
    "agent_architecture": {
        "name": "agent_architecture_default_v2",
        "version": "1.0.0",
    },
    "observability_configs": [],
    "question_groups": [],
    "extra": {
        "test": "test",
    },
})
print(response.json())


{'detail': 'Agent name Test Agent 2: examples/create-agent.ipynb is not unique'}


In [4]:
name_url_encoded = url_quote("Test Agent 2: examples/create-agent.ipynb")
agent_by_name = get(f"http://localhost:8000/api/v2/agents/by-name?name={name_url_encoded}").json()

# Get id to use in the next few examples
agent_id = agent_by_name["agent_id"]

In [5]:
from widget import DebugChatWidget

widget = DebugChatWidget(
    agent_id=agent_id,
    base_url="http://localhost:8000/api/v2",
)
widget

DebugChatWidget(agent_id='0772bd13-d54c-4e06-bef2-b956fa45e832', base_url='http://localhost:8000/api/v2', mess…

Received message: {'type': 'select_thread', 'thread_id': 'f3f98c89-cdd5-4f27-9e72-4907b95a0c47'}
Received message: {'type': 'select_thread', 'thread_id': '936d4c65-3d87-4f4a-9ad4-d6fb752b316b'}
Received message: {'type': 'select_thread', 'thread_id': '57ff5b5d-db6a-42b2-95d1-52725d28feef'}
Received message: {'type': 'select_thread', 'thread_id': 'e9aaa437-b7df-40a7-b225-21f69f0bd89d'}
Received message: {'type': 'select_thread', 'thread_id': 'f3f98c89-cdd5-4f27-9e72-4907b95a0c47'}
Received message: {'type': 'select_thread', 'thread_id': '936d4c65-3d87-4f4a-9ad4-d6fb752b316b'}
Received message: {'type': 'select_thread', 'thread_id': 'f3f98c89-cdd5-4f27-9e72-4907b95a0c47'}
Received message: {'type': 'new_thread'}
Received message: {'type': 'user_input', 'text': 'testing'}
[WS] Message complete, connection will be closed by server
Received message: {'type': 'select_thread', 'thread_id': '57ff5b5d-db6a-42b2-95d1-52725d28feef'}
Received message: {'type': 'select_thread', 'thread_id': '4412a6

In [ ]:
# Now list threads for the agent
specific_thread = get(f"http://localhost:8000/api/v2/threads?agent_id={agent_id}").json()
print(json.dumps(specific_thread, indent=2))

In [ ]:
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}").json()
print(json.dumps(specific_agent, indent=2))

In [ ]:
delete(f"http://localhost:8000/api/v2/agents/{agent_id}")

In [ ]:
# If we get the agent again, it should be gone
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}")
print(json.dumps({
    "status_code": specific_agent.status_code,
    "response": specific_agent.json(),
}, indent=2))
